# Praktikum Pertemuan 3: Classification

**Nama:** Hafidz Rizqullah Prasetya  
**NIM:** 24/535493/SV/24243  
**Kelas:** PL5A1  
**Dosen Pengampu:** Dr. Imam Fahrurrozi, S.T., M.Cs.

Notebook ini mendokumentasikan percobaan klasifikasi sesuai Modul Pertemuan 3 (Stroke Prediction Dataset + Tugas Loan Status Prediction). Jalankan sel secara berurutan di Google Colaboratory.

## 1. Impor Pustaka

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

print('pandas:', pd.__version__)
print('numpy:', np.__version__)
import sklearn
print('scikit-learn:', sklearn.__version__)
import matplotlib
print('matplotlib:', matplotlib.__version__)

## 2. Load Dataset Stroke
Dataset: https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset  
Mirror: https://raw.githubusercontent.com/ganjar87/data_science_practice/refs/heads/main/healthcare-dataset-stroke-data.csv

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/ganjar87/data_science_practice/refs/heads/main/healthcare-dataset-stroke-data.csv')
display(df.head())
display(df.tail())
print('shape:', df.shape)
print('columns:', list(df.columns))
print(df.info())

## 3. Exploratory Data Analysis (EDA)
### 3a. Deskripsi Statistik

In [ ]:
display(df.describe())
# df.describe() menunjukkan count bmi = 4909 (ada 201 missing), mean age 43.22, mean stroke 0.048

### 3b. Presentase Target Kelas

In [ ]:
data = df['stroke'].value_counts()
print(data)
data.plot(kind='pie', autopct='%.2f%%')
plt.title('Distribusi Kelas Stroke')
plt.ylabel('')
plt.show()
# Apakah distribusi seimbang? Tidak — 95.13% kelas 0, 4.87% kelas 1 (imbalanced)

### 3c. Histogram Variabel Numerik

In [ ]:
df.hist(figsize=(10,10))
plt.tight_layout()
plt.show()
# age agak normal sedikit skew kanan, avg_glucose_level & bmi right-skewed (outlier)

### 3d. Pengecekan Missing Value

In [ ]:
print(df.isnull().sum())
# Hanya kolom bmi yang memiliki 201 missing values

### 3e. Pengecekan Atribut Kategorikal

In [ ]:
df_X_tmp = df.drop(['id', 'stroke'], axis=1)
cats = df_X_tmp.select_dtypes(include=['object', 'bool']).columns
print(cats.tolist())
# gender, ever_married, work_type, Residence_type, smoking_status perlu encoding

## 4. Data Preprocessing
Langkah: pisahkan X/y, imputasi median bmi, encoding kategorikal, konversi array, split 70:30, StandardScaler (fit hanya di train).

In [ ]:
# --- Data Preprocessing dimulai ---
# Membuat X dan y
df_X = df.drop(['id', 'stroke'], axis=1)
df_y = df[['stroke']]

# Label encoding untuk y (sebenarnya sudah 0/1, tetap dilakukan sesuai modul)
le_y = LabelEncoder()
df_y = le_y.fit_transform(df_y['stroke'])

# Imputasi nilai hilang pada bmi dengan median
# Pandas otomatis menganggap 'N/A' sebagai NaN saat read_csv, jadi median valid
# Pastikan kolom bmi numerik (coerce jika masih object)
df_X['bmi'] = pd.to_numeric(df_X['bmi'], errors='coerce')
median_bmi = df_X['bmi'].median()
print('median bmi:', median_bmi)
df_X['bmi'] = df_X['bmi'].fillna(median_bmi)

# Categorical encoding dengan LabelEncoder
cats = df_X.select_dtypes(include=['object', 'bool']).columns
cat_features = list(cats.values)
print('cat_features:', cat_features)
le = LabelEncoder()
for i in cat_features:
    df_X[i] = le.fit_transform(df_X[i].astype(str))

# Simpan X dan y menjadi numpy arrays bertipe float
X = df_X.astype(float).values
y = df_y.astype(float)
print('X shape:', X.shape, 'y shape:', y.shape)
print('X[0:2]:', X[:2])
print('y[0:10]:', y[:10])

# Hold-out split 70:30
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print('X_train:', X_train.shape, 'X_test:', X_test.shape, 'y_train:', y_train.shape, 'y_test:', y_test.shape)

# Scaling (fit hanya pada X_train)
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
print('X_train scaled mean (first 3 cols):', X_train.mean(axis=0)[:3])
print('X_train scaled std (first 3 cols):', X_train.std(axis=0)[:3])
# --- Data Preprocessing selesai ---

### Pemeriksaan Hasil Preprocessing

In [ ]:
# a. Variabel X setelah konversi numerik
print('X[:5] =')
print(X[:5])

# b. Variabel y
print('y[:20] =', y[:20])
print('unique y:', np.unique(y))

# c/d. X_train dan X_test scaled
print('X_train[:3] =')
print(X_train[:3])
print('X_test[:3] =')
print(X_test[:3])

## 5. Modelling
Membandingkan 5 algoritma: Logistic Regression, KNN (k=10), Decision Tree (entropy), Random Forest, AdaBoost.

### 5a. Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Accuracy ', accuracy_score(y_test, y_pred))
print('Precision ', precision_score(y_test, y_pred, average='macro', zero_division=0))
print('Recall ', recall_score(y_test, y_pred, average='macro', zero_division=0))
print('Confusion matrix ')
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y_test))
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix - Logistic Regression')
plt.show()

print('F1 ', f1_score(y_test, y_pred, average='macro', zero_division=0))
print('coef :', model.coef_)
print('intercept:', model.intercept_)

### 5b. K-Nearest Neighbors (KNN, k=10)

In [ ]:
model = KNeighborsClassifier(n_neighbors=10)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Accuracy ', accuracy_score(y_test, y_pred))
print('Precision ', precision_score(y_test, y_pred, average='macro', zero_division=0))
print('Recall ', recall_score(y_test, y_pred, average='macro', zero_division=0))
print('Confusion matrix ')
print(confusion_matrix(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y_test))
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix - KNN (k=10)')
plt.show()

### 5c. Decision Tree (criterion='entropy')

In [ ]:
model = DecisionTreeClassifier(criterion='entropy', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Accuracy ', accuracy_score(y_test, y_pred))
print('Precision ', precision_score(y_test, y_pred, average='macro', zero_division=0))
print('Recall ', recall_score(y_test, y_pred, average='macro', zero_division=0))
print('Confusion matrix ')
print(confusion_matrix(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y_test))
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix - Decision Tree (entropy)")
plt.show()

### 5d. Random Forest

In [ ]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Accuracy ', accuracy_score(y_test, y_pred))
print('Precision ', precision_score(y_test, y_pred, average='macro', zero_division=0))
print('Recall ', recall_score(y_test, y_pred, average='macro', zero_division=0))
print('Confusion matrix ')
print(confusion_matrix(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y_test))
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix - Random Forest')
plt.show()

### 5e. AdaBoost

In [ ]:
model = AdaBoostClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Accuracy ', accuracy_score(y_test, y_pred))
print('Precision ', precision_score(y_test, y_pred, average='macro', zero_division=0))
print('Recall ', recall_score(y_test, y_pred, average='macro', zero_division=0))
print('Confusion matrix ')
print(confusion_matrix(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y_test))
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix - AdaBoost')
plt.show()

## 6. Tugas & Analisis — Loan Status Prediction
Dataset: https://www.kaggle.com/datasets/bhavikjikadara/loan-status-prediction  
Mirror GitHub: https://raw.githubusercontent.com/shrikant-temburwar/Loan-Prediction-Dataset/master/train.csv  
Tugas: tujuan dataset, definisi input/output, pembuatan 4 model (Logistic Regression, KNN, Decision Tree, Random Forest), tabel performa, analisis model terbaik.

### 6a. Penjelasan Dataset Loan
**Tujuan penggunaan dataset:** Dataset Loan Status Prediction digunakan untuk memprediksi apakah pengajuan pinjaman nasabah akan disetujui (Y) atau ditolak (N) berdasarkan profil peminjam (Gender, Married, Dependents, Education, Self_Employed, ApplicantIncome, CoapplicantIncome, LoanAmount, dll) dan riwayat kredit. Tujuannya membantu lembaga keuangan mengotomatisasi penilaian risiko kredit dan mengurangi kredit macet.

**Atribut input (fitur):** Gender, Married, Dependents, Education, Self_Employed, ApplicantIncome, CoapplicantIncome, LoanAmount, Loan_Amount_Term, Credit_History, Property_Area (Loan_ID dihapus sebagai identifier).  
**Atribut output (target):** Loan_Status (Y/N) dikonversi menjadi 1/0.

Dataset berisi 614 baris, 13 kolom, dengan missing value pada beberapa kolom kategorikal dan numerik.

### 6b. Load dan EDA Dataset Loan

In [ ]:
# Load dataset Loan (mirror GitHub karena Kaggle butuh download manual)
loan_url = 'https://raw.githubusercontent.com/shrikant-temburwar/Loan-Prediction-Dataset/master/train.csv'
loan_df = pd.read_csv(loan_url)
print('shape:', loan_df.shape)
display(loan_df.head())
display(loan_df.tail())
print(loan_df.info())
display(loan_df.describe(include='all'))
print('Loan_Status distribution:')
print(loan_df['Loan_Status'].value_counts())
print(loan_df['Loan_Status'].value_counts(normalize=True)*100)
print('missing values:')
print(loan_df.isnull().sum())
print('duplicates:', loan_df.duplicated().sum())

# Pie chart target
loan_df['Loan_Status'].value_counts().plot(kind='pie', autopct='%.2f%%')
plt.title('Distribusi Loan_Status')
plt.ylabel('')
plt.show()

# Histogram numerik
num_cols = loan_df.select_dtypes(include='number').columns.tolist()
print('numeric cols:', num_cols)
loan_df[num_cols].hist(figsize=(10,6))
plt.tight_layout()
plt.show()

### 6c. Preprocessing Dataset Loan

In [ ]:
# Preprocessing Loan — imputasi, encoding, split 70:30, scaling
# 1. Handle Dependents '3+' -> 3
loan_df['Dependents'] = loan_df['Dependents'].replace('3+', 3)
loan_df['Dependents'] = pd.to_numeric(loan_df['Dependents'], errors='coerce')

# 2. Imputasi missing value: median untuk numerik, mode untuk kategorikal
num_cols_loan = ['Dependents', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
cat_cols_loan = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']
for col in num_cols_loan:
    loan_df[col] = pd.to_numeric(loan_df[col], errors='coerce')
    loan_df[col] = loan_df[col].fillna(loan_df[col].median())
for col in cat_cols_loan:
    loan_df[col] = loan_df[col].fillna(loan_df[col].mode()[0])
print('after imputation missing:')
print(loan_df.isnull().sum())

# 3. Encoding kategorikal dan target
le_target = LabelEncoder()
y_loan = le_target.fit_transform(loan_df['Loan_Status'])  # Y=1, N=0
print('target mapping Y->', le_target.transform(['Y'])[0], 'N->', le_target.transform(['N'])[0])

X_loan = loan_df.drop(['Loan_ID', 'Loan_Status'], axis=1)
# LabelEncoder untuk tiap kolom kategorikal
for col in cat_cols_loan:
    X_loan[col] = LabelEncoder().fit_transform(X_loan[col].astype(str))
print('X_loan head:')
display(X_loan.head())
print('X_loan shape:', X_loan.shape, 'y_loan shape:', y_loan.shape)

# 4. Train-test split 70:30
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_loan, y_loan, test_size=0.3, random_state=42)
print('X_train_l:', X_train_l.shape, 'X_test_l:', X_test_l.shape)

# 5. Scaling (fit hanya pada train)
scaler_l = StandardScaler().fit(X_train_l)
X_train_l_scaled = scaler_l.transform(X_train_l)
X_test_l_scaled = scaler_l.transform(X_test_l)
print('scaled mean first 3:', X_train_l_scaled.mean(axis=0)[:3])
print('scaled std first 3:', X_train_l_scaled.std(axis=0)[:3])

### 6d. Modelling Loan — Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_l_scaled, y_train_l)
y_pred_l = model.predict(X_test_l_scaled)
print('Accuracy ', accuracy_score(y_test_l, y_pred_l))
print('Precision ', precision_score(y_test_l, y_pred_l, average='macro', zero_division=0))
print('Recall ', recall_score(y_test_l, y_pred_l, average='macro', zero_division=0))
print('F1 ', f1_score(y_test_l, y_pred_l, average='macro', zero_division=0))
print(confusion_matrix(y_test_l, y_pred_l))
cm = confusion_matrix(y_test_l, y_pred_l)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_target.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Loan - Logistic Regression')
plt.show()

### 6e. Modelling Loan — KNN

In [ ]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_l_scaled, y_train_l)
y_pred_l = model.predict(X_test_l_scaled)
print('Accuracy ', accuracy_score(y_test_l, y_pred_l))
print('Precision ', precision_score(y_test_l, y_pred_l, average='macro', zero_division=0))
print('Recall ', recall_score(y_test_l, y_pred_l, average='macro', zero_division=0))
print('F1 ', f1_score(y_test_l, y_pred_l, average='macro', zero_division=0))
print(confusion_matrix(y_test_l, y_pred_l))
cm = confusion_matrix(y_test_l, y_pred_l)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_target.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Loan - KNN k=5')
plt.show()

### 6f. Modelling Loan — Decision Tree

In [ ]:
model = DecisionTreeClassifier(criterion='entropy', random_state=42)
model.fit(X_train_l_scaled, y_train_l)
y_pred_l = model.predict(X_test_l_scaled)
print('Accuracy ', accuracy_score(y_test_l, y_pred_l))
print('Precision ', precision_score(y_test_l, y_pred_l, average='macro', zero_division=0))
print('Recall ', recall_score(y_test_l, y_pred_l, average='macro', zero_division=0))
print('F1 ', f1_score(y_test_l, y_pred_l, average='macro', zero_division=0))
print(confusion_matrix(y_test_l, y_pred_l))
cm = confusion_matrix(y_test_l, y_pred_l)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_target.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Loan - Decision Tree entropy')
plt.show()

### 6g. Modelling Loan — Random Forest

In [ ]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_l_scaled, y_train_l)
y_pred_l = model.predict(X_test_l_scaled)
print('Accuracy ', accuracy_score(y_test_l, y_pred_l))
print('Precision ', precision_score(y_test_l, y_pred_l, average='macro', zero_division=0))
print('Recall ', recall_score(y_test_l, y_pred_l, average='macro', zero_division=0))
print('F1 ', f1_score(y_test_l, y_pred_l, average='macro', zero_division=0))
print(confusion_matrix(y_test_l, y_pred_l))
cm = confusion_matrix(y_test_l, y_pred_l)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_target.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Loan - Random Forest')
plt.show()

### 6h. Tabel Performa Model (Accuracy, Precision, Recall)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

results = []
configs = [
    ('Logistic Regression', LogisticRegression(max_iter=1000)),
    ('KNN (k=5)', KNeighborsClassifier(n_neighbors=5)),
    ('Decision Tree', DecisionTreeClassifier(criterion='entropy', random_state=42)),
    ('Random Forest', RandomForestClassifier(random_state=42)),
]
for name, clf in configs:
    clf.fit(X_train_l_scaled, y_train_l)
    yp = clf.predict(X_test_l_scaled)
    acc = accuracy_score(y_test_l, yp)
    prec = precision_score(y_test_l, yp, average='macro', zero_division=0)
    rec = recall_score(y_test_l, yp, average='macro', zero_division=0)
    f1 = f1_score(y_test_l, yp, average='macro', zero_division=0)
    results.append([name, round(acc,4), round(prec,4), round(rec,4), round(f1,4)])
    print(f"{name}: acc={acc:.4f} prec={prec:.4f} rec={rec:.4f} f1={f1:.4f}")

perf_df = pd.DataFrame(results, columns=['model','accuracy','precision','recall','f1'])
display(perf_df)
# Analisis: model dengan accuracy/precision/recall tertinggi adalah yang terbaik. Biasanya Random Forest unggul karena ensemble.

## Catatan Screenshot
Semua screenshot hasil eksekusi cell di atas sudah dipasang pada laporan LaTeX (`laporan-3.tex`) sebagai Gambar ss-01 s/d ss-32 dan ss-tugas-01 s/d ss-tugas-08. Untuk laporan, jalankan notebook di Google Colab dan simpan output sebagai gambar ke folder `images/`.